# 02 - Lakeflow Declarative Pipeline: P&C Underwriting & Claims

**Project:** P&C Underwriting & Claims Analytics
**Source:** Wisconsin Local Government Property Insurance Fund (LGPIF)

## What this notebook does
Builds the full bronze -> silver -> gold medallion pipeline declaratively.
Unlike the Healthcare project's numbered-notebook + Job approach, this
project's core pipeline lives entirely inside a single Lakeflow Declarative
Pipeline: each function below defines one table, and the pipeline engine
infers execution order from which tables reference which.

## Important
This notebook is NOT run directly. It's attached to a Lakeflow ETL Pipeline
(Jobs & Pipelines -> Create -> ETL Pipeline), same process as the Healthcare
project's notebook 06, just used as the PRIMARY implementation here rather
than a supplementary demo.

## Tables in this pipeline
- bronze_property_fund
- silver_property_fund
- gold_loss_experience_by_risk_factor
- gold_loss_experience_by_entity_year
- gold_loss_ratio_summary

## Data quality note: EntityType mapping
Source documentation names six entity types (Village, City, County, Misc,
School, Town) but only explicitly numbers five of them (1-5), leaving "Town"
unassigned. Inferred 6 = Town as the only logical completion (one code, one
leftover name). Verified structurally: all six codes 1-6 appear in the data
with no gaps and nothing beyond 6, consistent with this mapping - but this
remains an inference from the data's structure, not a confirmed fact from an
explicit source data dictionary.

## Data quality validation results
All four DLT expectations (valid_policy_num, valid_year, non_negative_premium,
non_negative_claim) show zero violations across all 5,639 source rows -
consistent with this being a curated academic teaching dataset rather than
raw production data. Verified via the pipeline's Data Quality tab, not
assumed.

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.sql.functions import current_timestamp, col

bronze_schema = StructType([
    StructField("PolicyNum", IntegerType(), True),
    StructField("Year", IntegerType(), True),
    StructField("Premium", DoubleType(), True),
    StructField("Deduct", IntegerType(), True),
    StructField("BCcov", DoubleType(), True),
    StructField("Freq", IntegerType(), True),
    StructField("Fire5", IntegerType(), True),
    StructField("NoClaimCredit", IntegerType(), True),
    StructField("EntityType", IntegerType(), True),
    StructField("AlarmCredit", IntegerType(), True),
    StructField("BCClaim", DoubleType(), True),
])

@dp.table(
    name="bronze_property_fund",
    comment="Raw Wisconsin LGPIF policy-year records, ingested via Autoloader",
)
def bronze_property_fund():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .schema(bronze_schema)
        .load("/Volumes/main/property_casualty_ins/raw_data/WiscPropFund*.csv")
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )

The Delta Live Tables (DLT) module is not supported on this cluster.
 You should either create a new pipeline or use an existing pipeline to run DLT code.

---------------------------------------------------------------------------
ModuleNotFoundError                       Traceback (most recent call last)
File <command-7219481578963443>, line 1
----> 1 import dlt
      2 from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
      3 from pyspark.sql.functions import current_timestamp, col

ModuleNotFoundError: No module named 'dlt'

In [0]:
from pyspark.sql.functions import when, col, round as _round

@dlt.table(
    name="silver_property_fund",
    comment="Cleaned policy-year records: readable entity types, boolean risk flags, and computed severity metrics",
)
@dlt.expect_or_drop("valid_policy_num", "policy_num IS NOT NULL")
@dlt.expect_or_drop("valid_year", "policy_year BETWEEN 2000 AND 2020")
@dlt.expect("non_negative_premium", "premium >= 0")
@dlt.expect("non_negative_claim", "claim_amount >= 0")
def silver_property_fund():
    return (
        dlt.read_stream("bronze_property_fund")
        .withColumnRenamed("PolicyNum", "policy_num")
        .withColumnRenamed("Year", "policy_year")
        .withColumnRenamed("Premium", "premium")
        .withColumnRenamed("Deduct", "deductible")
        .withColumnRenamed("BCcov", "coverage_amount")
        .withColumnRenamed("Freq", "claim_count")
        .withColumnRenamed("BCClaim", "claim_amount")
        .withColumn("is_low_fire_class", col("Fire5") == 1)
        .withColumn("has_no_claim_credit", col("NoClaimCredit") == 1)
        .withColumn("has_claim", col("claim_count") > 0)
        .withColumn(
            "entity_type",
            when(col("EntityType") == 1, "Village")
            .when(col("EntityType") == 2, "City")
            .when(col("EntityType") == 3, "County")
            .when(col("EntityType") == 4, "Misc")
            .when(col("EntityType") == 5, "School")
            .when(col("EntityType") == 6, "Town")
            .otherwise("Unknown")
        )
        .withColumn(
            "avg_claim_severity",
            when(col("claim_count") > 0, _round(col("claim_amount") / col("claim_count"), 2)).otherwise(None)
        )
        .select(
            "policy_num", "policy_year", "premium", "deductible", "coverage_amount",
            "entity_type", "AlarmCredit", "is_low_fire_class", "has_no_claim_credit",
            "claim_count", "claim_amount", "has_claim", "avg_claim_severity",
        )
        .withColumnRenamed("AlarmCredit", "alarm_credit_pct")
    )

In [0]:
from pyspark.sql.functions import count as _count, sum as _sum, round as _round, expr, when, col

@dlt.table(
    name="gold_loss_experience_by_risk_factor",
    comment="Claim frequency and severity by underwriting risk factors (fire class, no-claim credit, alarm credit), for pricing and underwriting review. NOTE: avg_severity and median_claim_severity are both computed only among policy-years with at least one claim. The top segment's elevated claim_frequency_rate is driven primarily by two large, high-volume policies rather than a distortion in per-claim severity (avg_severity and median_claim_severity agree closely there) - see 99_pc_scratch_investigation for the underlying policy-level detail. Segments with policy_year_count below ~50 should be read cautiously; sample size is too small for severity estimates to be reliable.",
)
def gold_loss_experience_by_risk_factor():
    return (
        dlt.read("silver_property_fund")
        .groupBy("is_low_fire_class", "has_no_claim_credit", "alarm_credit_pct")
        .agg(
            _count("policy_num").alias("policy_year_count"),
            _sum("claim_count").alias("total_claims"),
            _sum("claim_amount").alias("total_claim_amount"),
            _round(_sum("claim_amount") / _sum("claim_count"), 2).alias("avg_severity"),
            _round(
                expr("percentile_approx(CASE WHEN has_claim THEN claim_amount ELSE NULL END, 0.5)"), 2
            ).alias("median_claim_severity"),
            _round(_sum("claim_count") / _count("policy_num"), 3).alias("claim_frequency_rate"),
        )
    )

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-7219481578963445>, line 3
      1 from pyspark.sql.functions import count as _count, sum as _sum, round as _round, expr, when, col
----> 3 @dlt.table(
      4     name="gold_loss_experience_by_risk_factor",
      5     comment="Claim frequency and severity by underwriting risk factors (fire class, no-claim credit, alarm credit), for pricing and underwriting review. NOTE: avg_severity and median_claim_severity are both computed only among policy-years with at least one claim (most policy-years have zero claims, so including them would trivially skew any severity metric toward zero). avg_severity is sensitive to a small number of high-volume policies (e.g. large multi-building accounts) that can dominate a segment's total - median_claim_severity is included as an outlier-resistant companion metric. See silver_property_fund f

In [0]:
@dp.table(
    name="gold_loss_experience_by_entity_year",
    comment="Claim frequency, severity, and premium trends by entity type and year, for portfolio-level trend analysis. NOTE: avg_severity at this grain (entity_type x policy_year) can be volatile due to small claim counts in some cells (e.g. Town 2008 had only 16 claims, Misc 2010 had 34) - a single large claim can swing a year's average several-fold. total_claim_amount is a more stable trend metric for this grain. School's 2010 total_claim_amount ($22.3M, ~3x any other year) is a genuine large aggregate loss, not a small-sample artifact (486 claims that year, not a small count) - underlying cause not identifiable from this dataset alone.",
)
def gold_loss_experience_by_entity_year():
    return (
        spark.read.table("silver_property_fund")
        .groupBy("entity_type", "policy_year")
        .agg(
            _count("policy_num").alias("policy_count"),
            _sum("premium").alias("total_premium"),
            _sum("claim_count").alias("total_claims"),
            _sum("claim_amount").alias("total_claim_amount"),
            _round(_sum("claim_amount") / _sum("claim_count"), 2).alias("avg_severity"),
        )
    )

In [0]:
@dp.table(
    name="gold_loss_ratio_summary",
    comment="Loss ratio (claims paid / premium collected) by entity type - the core profitability metric in P&C insurance",
)
def gold_loss_ratio_summary():
    return (
        spark.read.table("silver_property_fund")
        .groupBy("entity_type")
        .agg(
            _sum("premium").alias("total_premium"),
            _sum("claim_amount").alias("total_claim_amount"),
            _round(_sum("claim_amount") / _sum("premium") * 100, 1).alias("loss_ratio_pct"),
        )
    )